In [1]:
import sys
sys.path.append("../")

In [2]:
from src.data_loader import download_btc_data, save_to_csv, load_from_csv

df = load_from_csv("../data/btc_data.csv")

In [3]:
# Xem 5 dòng đầu tiên của Dataset
df.head()

,close,high,low,open,volume
Date,,,,,
2017-01-01,998.325012,1003.080017,958.698975,963.658020,147775008
2017-01-02,1021.750000,1031.390015,996.702026,998.617004,222184992
2017-01-03,1043.839966,1044.079956,1021.599976,1021.599976,185168000
2017-01-04,1154.729980,1159.420044,1044.400024,1044.400024,344945984
2017-01-05,1013.380005,1191.099976,910.416992,1156.729980,510199008


In [4]:
# Xem thông tin tổng quan về Dataset
df.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 3356 entries, 2017-01-01 to 2026-03-10
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   close   3356 non-null   float64
 1   high    3356 non-null   float64
 2   low     3356 non-null   float64
 3   open    3356 non-null   float64
 4   volume  3356 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 157.3 KB


In [5]:
# Xem thống kê mô tả của các cột số
df.describe()

,close,high,low,open,volume
count,3356.000000,3356.000000,3356.000000,3356.000000,3.356000e+03
mean,34503.469630,35174.696410,33746.667260,34484.267903,2.749855e+10
std,32490.632402,33020.579959,31913.371926,32490.719579,2.263863e+10
min,777.757019,823.307007,755.755981,775.177979,6.085170e+07
25%,8164.680420,8289.237549,7929.757568,8161.269165,1.087277e+10
50%,23138.622070,23518.148438,22716.174805,23120.910156,2.386838e+10
75%,55755.476562,57284.095703,53689.908203,55424.416992,3.816633e+10
max,124752.531250,126198.070312,123196.046875,124752.140625,3.509679e+11


In [6]:
df.columns

Index(['close', 'high', 'low', 'open', 'volume'], dtype='str')

In [7]:
print(df.tail())

                   close          high           low          open  \
Date                                                                 
2026-03-06  68136.492188  71378.570312  67757.820312  70842.156250   
2026-03-07  67272.593750  68515.164062  66969.257812  68136.687500   
2026-03-08  65969.781250  68177.789062  65639.195312  67272.500000   
2026-03-09  68402.382812  69474.945312  65858.007812  65969.585938   
2026-03-10  69607.765625  71249.648438  68443.148438  68443.148438   

                 volume  
Date                     
2026-03-06  43776962871  
2026-03-07  23258701211  
2026-03-08  33195080116  
2026-03-09  49499875378  
2026-03-10  49920614400  


In [8]:
import os

# Kiểm tra file có tồn tại không
print(os.path.exists("../results/metrics.json"))

True


In [9]:
feature_cols = [
        "close",
        "high"
    ]
test = df[feature_cols].values
print(test)

[[  998.32501221  1003.08001709]
 [ 1021.75        1031.39001465]
 [ 1043.83996582  1044.07995605]
 ...
 [65969.78125    68177.7890625 ]
 [68402.3828125  69474.9453125 ]
 [69607.765625   71249.6484375 ]]


In [10]:
df.columns

Index(['close', 'high', 'low', 'open', 'volume'], dtype='str')

In [11]:
from src.features import build_features

df_features_engineering = build_features(df)

In [12]:
df_features_engineering.columns

Index(['close', 'high', 'low', 'open', 'volume', 'log_returns',
       'returns_lag_1', 'returns_lag_3', 'returns_lag_7', 'rolling_mean_7',
       'rolling_std_7', 'rolling_mean_14', 'rolling_std_14', 'momentum_3',
       'momentum_7', 'momentum_14', 'target'],
      dtype='str')

In [13]:
df2 = load_from_csv()
df2 = build_features(df2)
print(df2.columns)

Index(['close', 'high', 'low', 'open', 'volume', 'log_returns',
       'returns_lag_1', 'returns_lag_3', 'returns_lag_7', 'rolling_mean_7',
       'rolling_std_7', 'rolling_mean_14', 'rolling_std_14', 'momentum_3',
       'momentum_7', 'momentum_14', 'target'],
      dtype='str')


In [14]:
feature_cols = [
        "log_returns",
        "volume",
        "rolling_mean_7",
        "rolling_std_7",
        "rolling_mean_14",
        "rolling_std_14",
        "momentum_3",
        "momentum_7",
        "momentum_14"
    ]

target_col = "target"
X = df2[feature_cols].values
y = df2[target_col].values
print(X.shape, y.shape)

(3342, 9) (3342,)


In [15]:
y

array([1, 1, 0, ..., 1, 1, 0], shape=(3342,))

In [16]:
from src.train import create_sequence_data, time_series_train_test_split
seq_length = 30
X_seq, y_seq = create_sequence_data(X, y, seq_length)
print(X_seq.shape, y_seq.shape)

(3312, 30, 9) (3312,)


In [17]:
X_train, X_val, X_test, y_train, y_val, y_test = time_series_train_test_split(X_seq, y_seq)
print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)


(2318, 30, 9) (2318,)
(497, 30, 9) (497,)
(497, 30, 9) (497,)


In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
    
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).view(-1,1)
    
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1,1)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=False)

In [19]:
X_train_tensor.type

<function Tensor.type>

In [20]:
2000 * 0.9

1800.0

In [21]:
X_train_tensor.shape[2]

9

In [22]:
df2['target'].value_counts()

target
1    1738
0    1604
Name: count, dtype: int64